# Análisis de ventas – TecnoAndes
Análisis de 150 transacciones (enero 2025 – octubre 2026, montos en USD) de una empresa que vende tecnología, oficina y software en 4 regiones.

**Preguntas que responde este notebook:** ingreso y margen total, desempeño por región, producto, categoría y vendedor, evolución mensual, y efecto de los descuentos sobre el margen.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:,.2f}'.format
plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Carga de datos

In [ ]:
archivo = 'practica_analisis_datos_empresariales.xlsx'
url = 'https://github.com/dicp24-cmyk/analisis-de-ventas/raw/main/' + archivo

try:
    df = pd.read_excel(archivo, sheet_name='Ventas')   # archivo local (mismo directorio)
except FileNotFoundError:
    df = pd.read_excel(url, sheet_name='Ventas')       # o directo desde GitHub

df.columns = ['fecha', 'region', 'vendedor', 'producto', 'categoria', 'unidades',
              'precio', 'descuento', 'costo_unit', 'ingreso', 'costo', 'margen', 'margen_pct']
df.head()

## 2. Revisión de calidad de los datos

In [ ]:
print('Filas y columnas:', df.shape)
print('Rango de fechas :', df['fecha'].min().date(), '->', df['fecha'].max().date())
print('Nulos           :', int(df.isna().sum().sum()))
print('Duplicados      :', int(df.duplicated().sum()))
df.dtypes

Los datos están completos y sin duplicados, así que no hace falta limpiar. Nota: **no hay datos de noviembre y diciembre de 2025**, por eso las comparaciones entre años usan solo enero–octubre.

## 3. Indicadores clave

In [ ]:
ingreso_total = df['ingreso'].sum()
margen_total = df['margen'].sum()

print(f'Ingreso total      : {ingreso_total:,.0f} USD')
print(f'Margen total       : {margen_total:,.0f} USD')
print(f'Margen % total     : {margen_total / ingreso_total:.2%}')
print(f'Ventas con pérdida : {(df["margen"] < 0).sum()}')
print(f'Ticket promedio    : {df["ingreso"].mean():,.0f} USD')

> El margen % total se calcula como margen total / ingreso total. No se promedia la columna `margen_pct`, porque eso daría el mismo peso a ventas pequeñas y grandes.

## 4. Ingreso por región

In [ ]:
region = df.groupby('region')['ingreso'].sum().sort_values()
ax = region.plot(kind='barh', color='#2a6f97')
ax.set_title('Ingreso por región (USD)')
ax.set_xlabel('USD'); ax.set_ylabel('')
plt.show()
region.sort_values(ascending=False)

## 5. Productos y categorías

In [ ]:
prod = df.groupby('producto').agg(unidades=('unidades', 'sum'),
                                  ingreso=('ingreso', 'sum'),
                                  margen=('margen', 'sum'))
prod['margen_pct'] = prod['margen'] / prod['ingreso']
prod.sort_values('unidades', ascending=False)

In [ ]:
cat = df.groupby('categoria').agg(ingreso=('ingreso', 'sum'), margen=('margen', 'sum'))
cat['margen_pct'] = cat['margen'] / cat['ingreso']

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
cat['ingreso'].plot(kind='bar', ax=axs[0], color='#2a6f97', rot=0)
axs[0].set_title('Ingreso por categoría (USD)')
(cat['margen_pct'] * 100).plot(kind='bar', ax=axs[1], color='#61a5c2', rot=0)
axs[1].set_title('Margen % por categoría')
for a in axs: a.set_xlabel('')
plt.tight_layout(); plt.show()
cat

**Hallazgo:** Tecnología es la categoría que más vende, pero Software tiene el mejor margen (cerca de 36 %, frente a 16 % de Tecnología). Laptop Pro vende más unidades, pero con un margen bajo (cerca de 15 %).

## 6. Vendedores

In [ ]:
vend = df.groupby('vendedor')[['ingreso', 'margen']].sum().sort_values('margen')
ax = vend['margen'].plot(kind='barh', color='#2a6f97')
ax.set_title('Margen por vendedor (USD)'); ax.set_ylabel('')
plt.show()
vend.sort_values('margen', ascending=False)

## 7. Evolución mensual

In [ ]:
df['anio'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month

mensual = df.pivot_table(index='mes', columns='anio', values='ingreso', aggfunc='sum')
ax = mensual.plot(marker='o')
ax.set_title('Ingreso mensual: 2025 vs 2026 (USD)')
ax.set_xlabel('Mes'); ax.set_xticks(range(1, 11))
ax.set_xlim(0.7, 10.3)
plt.show()

ene_oct = df[df['mes'] <= 10].groupby('anio')['ingreso'].sum()
crecimiento = ene_oct[2026] / ene_oct[2025] - 1
print(f'Ingreso ene–oct 2025: {ene_oct[2025]:,.0f} USD')
print(f'Ingreso ene–oct 2026: {ene_oct[2026]:,.0f} USD')
print(f'Crecimiento         : {crecimiento:.1%}')

q3_2026 = df[(df['fecha'] >= '2026-07-01') & (df['fecha'] <= '2026-09-30')]['ingreso'].sum()
print(f'Ingreso 3T 2026     : {q3_2026:,.0f} USD')

**Tendencia:** el ingreso crece alrededor de 39 % frente al mismo período de 2025. 2025 fue muy irregular (caídas en agosto y septiembre, pico en octubre) y 2026 se mantiene más alto y estable desde mayo.

## 8. Descuentos y margen

In [ ]:
desc = df.groupby('descuento').agg(ventas=('ingreso', 'size'),
                                   ingreso=('ingreso', 'sum'),
                                   margen=('margen', 'sum'))
desc['margen_pct'] = desc['margen'] / desc['ingreso']

ax = (desc['margen_pct'] * 100).plot(kind='bar', color='#2a6f97', rot=0)
ax.set_title('Margen % según nivel de descuento')
ax.set_xlabel('Descuento'); ax.set_ylabel('Margen %')
ax.set_xticklabels([f'{d:.0%}' for d in desc.index])
plt.show()

con_desc = df[df['descuento'] > 0]
print(f'Descuento promedio (solo ventas con descuento): {con_desc["descuento"].mean():.1%}')
print(f'Ventas con descuento: {len(con_desc)} de {len(df)}')
desc

In [ ]:
perdidas = df[df['margen'] < 0]
perdidas[['fecha', 'region', 'vendedor', 'producto', 'unidades', 'descuento', 'margen']]

## 9. Conclusiones y recomendación

- **Ingreso total:** 1.902.425 USD, con un margen de 23,2 %. La región **Sur** concentra el mayor ingreso (cerca de 634 mil USD).
- **Software** (Licencia CRM y Software Contable) es lo más rentable. **Laptop Pro** vende mucho pero deja poco margen.
- **Descuentos:** con descuentos de 25 % o 30 % el margen cae a cerca de 7 %, y las 6 ventas con pérdida tuvieron todas un 30 % de descuento.

**Recomendación a la gerencia:** poner un tope de descuento de 20 % (o exigir aprobación para descuentos mayores) y enfocar la venta en Software, donde el margen es más alto.